# Engine Tuning Lab — Temperature, Top-P, Top-K

**Program:** Brilian Sistem Informasi Bootcamp  
**Session 25:** Prompt Optimization

This notebook helps you understand how decoding parameters (Temperature, Top-P, Top-K) affect Gemini's output.

## 1. Setup

> ⚠️ **IMPORTANT — Never hardcode your API key.** Always use Google Colab Secrets.

**How to set it up:**
1. Click the 🔑 icon on the left sidebar in Colab
2. Add a secret named `GOOGLE_API_KEY` with your Gemini API key from [Google AI Studio](https://aistudio.google.com/)
3. Toggle **Notebook access** ON


In [ ]:
!pip install -q google-generativeai


In [ ]:
import google.generativeai as genai
from google.colab import userdata
import textwrap

# 🔐 Always retrieve secrets from Colab Secrets — never hardcode
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API configured successfully.")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please ensure GOOGLE_API_KEY is set in Colab Secrets.")


## 2. Helper Function & Base Prompt

The base prompt below is used for ALL experiments so we can isolate the effect of each parameter.


In [ ]:
def generate_with_tuning(prompt, temp=1.0, top_p=0.95, top_k=40):
    """Generate text using specific tuning parameters."""
    model = genai.GenerativeModel('gemini-2.5-flash')

    config = genai.types.GenerationConfig(
        temperature=temp,
        top_p=top_p,
        top_k=top_k,
    )

    response = model.generate_content(prompt, generation_config=config)

    print(f"--- Parameters: Temp={temp}, Top-P={top_p}, Top-K={top_k} ---")
    print(textwrap.fill(response.text, width=80))
    print("\n" + "=" * 50 + "\n")


# 🚗 Standard prompt — Mitsubishi context
test_prompt = (
    "Jelaskan dalam 3-4 kalimat mengapa Mitsubishi Xpander cocok "
    "untuk keluarga muda di Indonesia."
)


## 3. Experiment 1 — Temperature

Temperature controls **randomness**. Lower values (near 0) produce deterministic, repetitive output. Higher values (up to 2.0) produce more creative, diverse responses.


In [ ]:
# Low Temperature — Conservative, focused
generate_with_tuning(test_prompt, temp=0.1)

# Mid Temperature — Balanced
generate_with_tuning(test_prompt, temp=0.7)

# High Temperature — Creative, diverse
generate_with_tuning(test_prompt, temp=1.5)


### 🔍 Observe
- Apakah jawaban di Temp=0.1 lebih konsisten jika dijalankan berulang?
- Output mana yang paling "kreatif"?
- Output mana yang paling aman untuk dipakai di materi marketing resmi?


## 4. Experiment 2 — Top-P (Nucleus Sampling)

Top-P selects the smallest set of tokens whose cumulative probability ≥ P. Lower Top-P narrows the model to only the most likely candidates.


In [ ]:
# Low Top-P — Very focused vocabulary
generate_with_tuning(test_prompt, temp=1.0, top_p=0.1)

# High Top-P — Diverse vocabulary
generate_with_tuning(test_prompt, temp=1.0, top_p=0.99)


### 🔍 Observe
- Apakah variasi kosakata berubah signifikan?
- Output mana yang lebih "ekspresif"?


## 5. Experiment 3 — Top-K

Top-K limits the model to the K most probable next tokens. Top-K = 1 is greedy decoding (always pick the single most likely token).


In [ ]:
# Very low Top-K — greedy (deterministic)
generate_with_tuning(test_prompt, temp=1.0, top_k=1)

# High Top-K — variety
generate_with_tuning(test_prompt, temp=1.0, top_k=100)


## 6. Experiment 4 — Combined Configurations

Run the three "preset" configurations side-by-side. This mirrors the table in the slides.


In [ ]:
configs = [
    {"label": "A — Safe",     "temp": 0.2, "top_p": 0.5, "top_k": 20},
    {"label": "B — Balanced", "temp": 0.7, "top_p": 0.9, "top_k": 50},
    {"label": "C — Creative", "temp": 1.0, "top_p": 1.0, "top_k": 100},
]

for cfg in configs:
    print(f"\n>>> Config {cfg['label']}")
    generate_with_tuning(
        test_prompt,
        temp=cfg["temp"],
        top_p=cfg["top_p"],
        top_k=cfg["top_k"],
    )


## ✅ Recap

| Parameter | Effect |
|---|---|
| **Temperature ↑** | More creative & diverse, less stable |
| **Top-P ↑** | Larger candidate pool, richer vocabulary |
| **Top-K ↑** | More token variety, potentially noisy |

**Rule of thumb for Mitsubishi business use cases:**
- Internal reports / official spec sheet → Safe (temp 0.2, top_p 0.5)
- Customer-facing email / chat reply → Balanced (temp 0.7, top_p 0.9)
- Marketing copy / social caption ideation → Creative (temp 1.0, top_p 1.0)
